# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/8(목) 오전 · 설정 분리 · 알림 연동 · 하나로 잇기 — 실습</mark>

어제까지 경보를 요약하고(`event_summarizer.py`), 보고서를 만들었습니다(`report_generator.py`). 그런데 지금은 보고서 하나를 만들려면 **셀 여러 개를 순서대로** 눌러야 하고, 모델 이름 같은 값이 **코드 곳곳에** 박혀 있습니다.

오늘 오전에는 조각을 하나로 묶습니다. 끝나면 터미널에서 **명령 한 번**으로 이렇게 돕니다.

```
$ python pipeline.py
[보고서] daily_report_20261008.md 저장 · 사람 확인 필요 1건
```

그리고 알림 서버 터미널에는 `[알림 받음] [보고서] daily_report_20261008.md 저장 …` 이 찍힙니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저

1. 이 파일을 어제와 같은 `security-agent-toolkit/agent_core` 폴더에 넣습니다.
2. 오른쪽 위 **커널 선택**에서 어제와 같은 **각자 만든 가상환경**을 고릅니다.
3. 아래 **위치 확인 셀**과 **준비 셀 넷**을 차례로 실행합니다. 어제를 못 끝냈어도 여기서부터 할 수 있습니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
1. 한도를 넘으면 답을 받지 못합니다. 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
2. 오전에 LLM 을 부르는 문제는 <b>4-2 · 4-3 · 4-5</b> 뿐입니다. 셋 다 한 번 실행에 두 번 부릅니다.<br>
3. 한도에 걸리면 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
4. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 오늘 오전의 순서

| 교시 | 무엇 | 만드는 파일 |
|---|---|---|
| 2교시 | 설정 분리 — 바뀌는 값은 파일 하나에 | `config.json` |
| 3교시 | 알림 연동 — 실패해도 멈추지 않게 | `notifier.py` |
| 4교시 | 하나로 잇기 — 명령 한 번에 끝까지 | `pipeline.py` |

### 0.3 문제를 푸는 법

1. 문제 앞의 **🐍 문법 상자**를 먼저 읽습니다. 그 문제에서 쓰는 문법이 있습니다.
2. 코드 셀의 번호 주석이 무엇을 쓸지 알려 줍니다. 주석에 적힌 **변수 이름을 그대로** 씁니다.
3. 막히면 **💡 힌트**를 보고, 그래도 막히면 노트북 맨 아래 「정답」을 봅니다. 도전 문제는 안 풀고 넘어가도 됩니다.
4. `NameError` 가 나오면 위쪽 셀을 건너뛴 것입니다.

In [ ]:
# 위치 확인 셀 — 그대로 실행합니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                                        # 다섯 칸 위까지 찾는다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 리턴한다
        folder = os.path.dirname(folder)                      # 한 칸 위 폴더로 올라간다
    return None                                               # None 을 리턴하고 여기서 끝낸다


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())                           # 노트북이 도는 폴더
print("지금 쓰는 파이썬     :", sys.executable)                      # 커널로 고른 파이썬 — 내 가상환경이어야 한다

if ENV_PATH:                                                  # .env 를 찾았으면
    print("키 파일(.env)         :", ENV_PATH)                   # 찾은 .env 의 위치
    print("준비 완료")
else:                                                         # 아니면
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 새 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) 아침 과제 6 의 「.env 옮기기」를 했는지 봅니다")

아래 네 셀도 실행합니다. 어제 만든 `llm_client.py` · `event_summarizer.py` · `report_generator.py` 를 정답과 같은 내용으로 다시 만들고(이미 있으면 같은 내용으로 덮어씁니다), 오늘 쓰는 경보 6건을 저장합니다.

In [ ]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 리턴한다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None                                               # None 을 리턴하고 여기서 끝낸다


def read_api_key():                                           # .env 에서 키를 읽어 리턴하는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 리턴한다


def call_llm(question):                                       # 질문을 받아 답 문장을 리턴하는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 리턴한다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 리턴한다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 리턴한다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None                                           # None 을 리턴하고 여기서 끝낸다

In [ ]:
%%writefile event_summarizer.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

SIZE = 6                                                      # 한 묶음에 넣을 경보 수
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "


def summarize_batch(batch):                                   # 묶음 하나를 LLM 으로 요약하는 함수
    # 묶음을 JSON 문자열로 바꿔 지시와 함께 보낸다
    text = call_llm(instruction + json.dumps(batch, ensure_ascii=False))
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result is None:                                        # JSON 으로 읽지 못했으면
        return []                                             # 실패하면 빈 리스트 — 다음 묶음은 계속한다
    return result                                             # 요약 리스트를 리턴한다


def summarize_events(events):                                 # 경보 전체를 묶음으로 나눠 요약하는 함수
    summaries = []                                            # 요약을 모을 빈 리스트
    for start in range(0, len(events), SIZE):                 # 0, 6, 12 … 묶음의 시작 번호
        # 묶음 하나를 요약해 한 건씩 꺼낸다
        for item in summarize_batch(events[start:start + SIZE]):
            summaries.append(item)                            # 요약 하나를 모은다
    return summaries                                          # 모은 요약을 리턴한다


def sort_by_risk(summaries):                                  # 요약을 high → medium → low 순서로 정렬하는 함수
    sorted_summaries = []                                     # 위험도순으로 모을 빈 리스트
    for level in ["high", "medium", "low"]:                   # 이 순서대로 모은다
        for s in summaries:                                   # 요약을 하나씩 꺼낸다
            if s["risk_level"].lower() == level:              # 소문자로 맞춰 지금 위험도와 같으면
                sorted_summaries.append(s)                    # 정렬 결과에 넣는다
    return sorted_summaries                                   # 정렬한 요약을 리턴한다

In [ ]:
%%writefile report_generator.py
from llm_client import call_llm                               # llm_client.py 의 call_llm 을 꺼내 쓴다

# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "


def make_lines(sorted_summaries):                             # 정렬된 요약을 목록 줄로 만드는 함수
    lines = ""                                                # 목록 줄을 모을 빈 문자열
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 「- [HIGH] E01 요약」 한 줄을 붙인다
        lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
    return lines                                              # 목록 줄을 리턴한다


def make_overview(sorted_summaries):                          # 정렬된 요약으로 총평을 받는 함수
    # 요약 줄을 지시와 함께 보내 총평을 받는다
    text = call_llm(overview_instruction + make_lines(sorted_summaries))
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        return "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 실패를 숨기지 않고 알린다
    return text.strip()                                       # 앞뒤 공백을 지운 총평을 리턴한다


def build_report(sorted_summaries, overview, today):          # 틀 · 숫자는 코드로, 총평은 받은 문장으로 보고서를 만드는 함수
    total = len(sorted_summaries)                             # 전체 건수 — 코드가 센다
    high_count = 0                                            # high 건수. 0 에서 시작
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        if s["risk_level"].lower() == "high":                 # 소문자로 맞춰 high 인지 본다
            high_count = high_count + 1                       # high 하나를 센다

    warning = ""                                              # 기본은 경고 없음
    if high_count >= 3:                                       # high 가 3건 이상이면
        # 경고 줄 — 보고서 맨 위 「한눈에 보기」에 들어간다
        warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"

    report = f"# 야간 보안 관제 보고 ({today})\n\n"                   # 제목 — 날짜만 바뀐다
    # 건수 줄과 경고 줄 — 코드가 센 값
    report = report + "## 한눈에 보기\n" + f"- 처리한 경보: {total}건 (high {high_count}건)\n" + warning
    report = report + "\n## 총평\n" + overview + "\n"           # 총평 — LLM 이 쓴 문장
    # 정렬한 목록 줄
    report = report + "\n## 건별 내역 (위험한 것부터)\n" + make_lines(sorted_summaries)
    return report                                             # 완성한 보고서 문자열을 리턴한다


def save_report(report, today):                               # 보고서를 날짜가 든 파일로 저장하는 함수
    filename = f"daily_report_{today.replace('-', '')}.md"    # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md
    with open(filename, "w", encoding="utf-8") as f:          # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
        f.write(report)                                       # 보고서 문자열을 파일에 쓴다
    return filename                                           # 저장한 파일 이름을 리턴한다

In [ ]:
# 준비 셀 — 그대로 실행합니다. 오늘 쓰는 경보 6건을 파일로 저장합니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

events_1008 = [                                               # 7교시 「다른 날」로 쓸 경보 6건
    {"id": "F01", "rule": "brute_force", "target": "admin", "time": "02:10", "detail": "203.0.113.9 에서 admin 로그인 실패 7회 뒤 성공"},
    {"id": "F02", "rule": "failed_login", "target": "kim01", "time": "09:05", "detail": "kim01 로그인 실패 1회"},
    {"id": "F03", "rule": "failed_login", "target": "lee.yh", "time": "09:40", "detail": "lee.yh 로그인 실패 1회 뒤 바로 성공"},
    {"id": "F04", "rule": "config_changed", "target": "firewall", "time": "18:00", "detail": "점검 시간에 ops.kim 이 방화벽 규칙 1개 수정"},
    {"id": "F05", "rule": "failed_login", "target": "park.js", "time": "10:12", "detail": "park.js 로그인 실패 1회"},
    {"id": "F06", "rule": "new_ip_login", "target": "jung.hw", "time": "08:55", "detail": "jung.hw 가 사내 다른 층 IP 에서 로그인"},
]

with open("events_1008.json", "w", encoding="utf-8") as f:        # 오늘 처리할 경보
    json.dump(events_1008, f, ensure_ascii=False, indent=2)   # 내일 경보를 파일에 저장한다

print(f"경보 {len(events_1008)}건을 저장했습니다")                      # 몇 건 저장했는지 본다

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 설정 분리 — config.json</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **설정 분리 (configuration)** | 코드와 설정을 왜 다른 파일에 두나 |
| **하드코딩 (hard-coding)** | 값을 코드에 직접 박아 두면 무엇이 불편한가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 설정 분리 →
- 하드코딩 →

### 왜 필요한가

1. 지금 모델 이름은 `llm_client.py` 안에, 날짜는 노트북 셀 안에 박혀 있습니다. 하나 바꾸려면 **코드를 뒤져야** 합니다.
2. 바뀔 수 있는 값을 **`config.json` 한 파일**로 모읍니다. 코드는 그대로 두고 이 파일만 바꿉니다.
3. 이미 한 번 해 봤습니다 — API 키를 `.env` 로 뺀 것입니다. **비밀은 `.env`, 비밀이 아닌 설정은 `config.json`** 에 둡니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 바뀌는 값은 config.json 에, 읽는 법은 json.load</mark>

```python
with open("config.json", encoding="utf-8") as f:
    config = json.load(f)                 # 읽으면 딕셔너리가 된다

print(config["model"])                    # gemini-3.5-flash-lite
```

| 키 | 값 | 무엇을 정하나 |
|---|---|---|
| `model` | `gemini-3.5-flash-lite` | 어느 LLM 을 부르나 |
| `approve_severity` | `high` | 어느 위험도부터 **사람에게 확인**받나 — 값이 아니라 **정책**이다 |
| `report_folder` | `reports` | 보고서를 어느 폴더에 모으나 |
| `webhook_url` | `http://127.0.0.1:5001/alert` | 알림을 어디로 보내나 |

| 파일 | 담는 것 | 깃허브에 |
|---|---|---|
| `.env` | 비밀 — API 키 | 올리지 않는다 |
| `config.json` | 설정 — 누가 언제 바꿨는지 남아야 하는 값 | 올린다 |

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
config = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}

print(config["approve_severity"])
print(len(config))
```

막히면 바로 위 `2.1 바뀌는 값은 config.json 에, 읽는 법은 json.load` 설명을 다시 봅니다.

In [ ]:
config = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}

print(config["approve_severity"])
print(len(config))

✅ `high` 와 `2` — 키가 두 개인 딕셔너리입니다

#### 🐍 문법 상자 · 파이썬 딕셔너리와 JSON 파일은 조금 다릅니다

```python
import json
print(json.dumps({"on": True, "url": None}))
# {"on": true, "url": null}
```

| 파이썬 | JSON 파일 | 차이 |
|---|---|---|
| `True` · `False` | `true` · `false` | 소문자 |
| `None` | `null` | 이름이 다르다 |
| `'작은따옴표'` 도 된다 | `"큰따옴표"` 만 된다 | `json.dump` 가 알아서 바꾼다 |

⚠ JSON 파일에는 주석(`#`)을 쓸 수 없습니다. 설명은 코드나 노트북에 둡니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · config.json 만들기</font></h3></td></tr></table>

키 두 개가 들어 있는 `config` 에 나머지 두 키를 더하고, `config.json` 파일로 저장하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `config.json 을 저장했습니다 · 키 4 개` — 왼쪽 목록에 `config.json` 이 생깁니다 |

**💡 힌트**

1. 딕셔너리에 키를 더할 때는 `config["키"] = 값` 입니다.
2. 저장은 10/7 오전에 쓴 `json.dump(…, f, ensure_ascii=False, indent=2)` 와 같습니다.
3. 저장한 파일을 왼쪽 목록에서 열어 네 줄이 보이는지 봅니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}   # 모델 이름 · 사람에게 물을 위험도
# ── 여기까지 ──

# 1. config 의 "report_folder" 키에 "reports" 를 담으세요


# 2. config 의 "webhook_url" 키에 "http://127.0.0.1:5001/alert" 를 담으세요


# 3. config 를 "config.json" 파일에 저장하세요 (json.dump, ensure_ascii=False, indent=2)


print("config.json 을 저장했습니다 · 키", len(config), "개")           # 키가 몇 개인지 본다

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · config.json 읽어서 값 꺼내기</font></h3></td></tr></table>

방금 저장한 `config.json` 을 읽어 모델 이름과 알림 주소를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `모델: gemini-3.5-flash-lite` · `알림 주소: http://127.0.0.1:5001/alert` |

**💡 힌트**

1. 읽기는 `json.load(f)` 입니다. 읽으면 딕셔너리가 됩니다.
2. 값은 `config["키"]` 로 꺼냅니다.
3. `KeyError` 가 나면 키 이름의 철자를 2-2 와 맞춰 봅니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# 1. "config.json" 을 읽기로 열어 config 에 담으세요


# 2. config 에서 모델 이름과 알림 주소를 꺼내 출력하세요


#### 🐍 문법 상자 · `in` 은 딕셔너리의 **키**를 봅니다

```python
config = {"model": "gemini-3.5-flash-lite"}
print("model" in config, "webhook_url" in config, "webhook_url" not in config)
# True False True
```

| 쓰는 것 | 결과 | 뜻 |
|---|---|---|
| `"model" in config` | `True` | 그 **키**가 있다 |
| `"webhook_url" not in config` | `True` | 그 키가 없다 |
| 반복 안의 `return` | 그 자리에서 함수가 끝난다 | 빠진 키 하나를 찾으면 더 보지 않는다 |

⚠ `"gemini-3.5-flash-lite" in config` 는 `False` 입니다. `in` 은 값이 아니라 키를 봅니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 설정을 검사하며 읽기</font></h3></td></tr></table>

꼭 있어야 하는 키 네 개가 다 있는지 검사하는 `load_config` 함수를 완성하시오. 빠진 키가 있으면 알려 주고 `None` 을 리턴합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 첫 줄은 설정 딕셔너리, 그다음 `[설정 오류] 빠진 키: report_folder` 와 `None` |

**💡 힌트**

1. `for key in REQUIRED:` 로 키를 하나씩 봅니다.
2. 없는지는 `not in` 으로 봅니다.
3. 빠진 키를 하나 찾으면 바로 `return None` 합니다. 나머지는 더 보지 않습니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
REQUIRED = ["model", "approve_severity", "report_folder", "webhook_url"]   # 꼭 있어야 하는 키 네 개
broken = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}   # 키 두 개가 빠진 설정
with open("config_broken.json", "w", encoding="utf-8") as f:  # 연습용 — 키가 빠진 설정 파일
    json.dump(broken, f)                                      # 빠진 설정을 저장한다
# ── 여기까지 ──


def load_config(path):                                        # 설정을 읽고 빠진 키가 없는지 검사하는 함수
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    # 1. REQUIRED 의 키를 하나씩 꺼내, config 에 없으면 "[설정 오류] 빠진 키:" 와 키 이름을 출력하고 None 을 return 하세요

    # 2. 다 있으면 config 를 return 하세요



print(load_config("config.json"))                             # 키가 다 있는 설정 — 딕셔너리가 나온다
print(load_config("config_broken.json"))                      # 키가 빠진 설정 — None 이 나온다

#### 🐍 문법 상자 · 위험도는 숫자로 바꿔 비교합니다

```python
RANK = {"low": 1, "medium": 2, "high": 3}
print(RANK["high"] >= RANK["medium"])
print("high" >= "medium")
# True
# False
```

| 비교 | 결과 | 이유 |
|---|---|---|
| `RANK["high"] >= RANK["medium"]` | `True` | 3 ≥ 2 |
| `"high" >= "medium"` | `False` | 글자끼리는 **사전 순서**로 비교한다. h 가 m 보다 앞이다 |

⚠ 글자를 그대로 비교하면 뜻과 상관없는 결과가 나옵니다. 순위를 사전(`RANK`)으로 숫자로 바꾼 뒤 비교합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 「high 이상」인지 판정하기</font></h3></td></tr></table>

위험도가 설정의 `approve_severity` **이상**이면 `True` 를 리턴하는 `needs_approval` 함수를 완성하시오. 사람에게 확인받을 경보를 고르는 함수입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `low False` · `medium False` · `High True` |

**💡 힌트**

1. 글자끼리 비교하지 않고 `RANK` 로 숫자를 꺼내 비교합니다.
2. `High` 처럼 대문자가 섞여 와도 되도록 `.lower()` 합니다.
3. 「이상」은 `>=` 입니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
RANK = {"low": 1, "medium": 2, "high": 3}                    # 위험도 순위 — 클수록 위험하다
# ── 여기까지 ──


def needs_approval(severity, config):                         # 사람에게 확인받을 위험도인지 판정하는 함수
    # 1. level 이라는 변수에 RANK 에서 severity.lower() 의 순위를 꺼내 담으세요

    # 2. limit 이라는 변수에 RANK 에서 config["approve_severity"] 의 순위를 꺼내 담으세요

    # 3. level 이 limit 보다 크거나 같은지(>=)를 return 하세요



for severity in ["low", "medium", "High"]:                    # 위험도 셋으로 시험한다 — 대문자가 섞여 있다
    print(severity, needs_approval(severity, config))         # 위험도와 판정 결과

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-6 · 설정만 바꿔 결과 바꾸기</font></h3></td></tr></table>

`approve_severity` 를 `medium` 으로 바꾼 설정 파일을 하나 더 만들고, 두 설정으로 각각 「사람에게 확인받을 경보」가 몇 건인지 세시오. **함수는 고치지 않습니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `config.json : 2 건` · `config_medium.json : 4 건` |

**💡 힌트**

1. `config.json` 자체는 바꾸지 않습니다. 새 파일로 저장합니다.
2. `for path in ["config.json", "config_medium.json"]:` 로 두 파일을 차례로 읽습니다.
3. 코드는 같고 설정만 다른데 결과가 바뀝니다. 이것이 설정 분리의 쓸모입니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
RANK = {"low": 1, "medium": 2, "high": 3}                     # 위험도 순위 — 클수록 위험하다


def needs_approval(severity, config):                         # 문제 2-5 의 함수
    # 순위가 기준 이상이면 True
    return RANK[severity.lower()] >= RANK[config["approve_severity"]]


severities = ["low", "medium", "high", "medium", "low", "high"]   # 오늘 판정한 위험도라고 가정합니다
# ── 여기까지 ──

# 1. config.json 을 읽어 config 에 담고, approve_severity 를 "medium" 으로 바꾼 것을 "config_medium.json" 으로 저장하세요
# 2. 두 파일을 하나씩 읽어, needs_approval 이 True 인 개수를 세세요
# 3. 「config.json : 2건」 · 「config_medium.json : 4건」 처럼 출력하세요


### 📋 2교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 설정을 저장한다 | `json.dump(config, f, ensure_ascii=False, indent=2)` |
| 설정을 읽는다 | `config = json.load(f)` → `config["model"]` |
| 빠진 키를 잡는다 | `if key not in config: return None` |
| 위험도를 비교한다 | `RANK[severity.lower()] >= RANK[config["approve_severity"]]` |

값은 파일에, 로직은 코드에 둡니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 알림 연동 — 실패해도 멈추지 않게</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **웹훅 알림** | 슬랙 · 팀즈 같은 메신저에 알림이 오는 방식 |
| **장애 허용 (fault tolerance)** | 한 부분에 장애가 나도 전체가 멈추지 않게 하려면 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 웹훅 알림 →
- 장애 허용 →

### 왜 필요한가

1. 보고서가 만들어져도 아무도 모르면 소용없습니다. 끝나면 **담당자에게 알림**을 보냅니다.
2. 알림은 10/2 에 만든 웹훅 서버로 보냅니다. 실무에서는 같은 자리에 슬랙 · 팀즈 · 메일 주소가 들어갑니다. **주소는 `config.json` 에서만 바꾸면** 됩니다.
3. 알림 서버가 꺼져 있을 수 있습니다. 알림 하나 실패했다고 **보고서까지 날아가면 안 됩니다.** 9/28 에 배운 `try` / `except` 로 버팁니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 알림 서버를 켜 둡니다</mark>

1. 아래 셀을 실행해 `alert_server.py` 를 만듭니다.
2. 왼쪽 목록에서 **`agent_core` 폴더를 오른쪽 클릭 › Open in Integrated Terminal** 을 누릅니다.
3. 💻 터미널에 입력합니다.

```bash
python alert_server.py
```

4. `Running on http://127.0.0.1:5001` 이 보이면 **그 터미널은 그대로 둡니다.** 알림이 오면 여기에 `[알림 받음]` 이 찍힙니다. 끌 때는 `Ctrl + C` 입니다.

| 이런 글이 보이면 | 이유 | 할 일 |
|---|---|---|
| `Address already in use` | 10/2 에 켠 서버가 아직 5001 번을 쓰고 있다 | 그 터미널을 찾아 `Ctrl + C` |
| `No module named 'flask'` | 터미널에 가상환경이 켜지지 않았다 | 터미널을 닫고 2번부터 다시 |

In [ ]:
%%writefile alert_server.py
from flask import Flask, request                              # 10/2 에 쓴 웹 서버 도구

app = Flask(__name__)                                         # 서버를 하나 만든다


@app.route("/alert", methods=["POST"])                        # /alert 로 POST 가 오면 아래 함수가 받는다
def alert():                                                  # 알림을 받는 함수
    data = request.get_json()                                 # 보낸 JSON 을 딕셔너리로 꺼낸다
    print("[알림 받음]", data["text"])                         # 터미널에 알림 내용을 보여 준다
    return {"ok": True}                                       # 받았다고 답한다 (상태 코드 200)


app.run(port=5001)                                            # 5001 번 포트에서 기다린다

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
try:
    number = int("abc")
    print("바꿨습니다")
except ValueError:
    print("숫자가 아닙니다")

print("다음 줄로 갑니다")
```

막히면 바로 위 `3.1 알림 서버를 켜 둡니다` 설명을 다시 봅니다.

In [ ]:
try:
    number = int("abc")
    print("바꿨습니다")
except ValueError:
    print("숫자가 아닙니다")

print("다음 줄로 갑니다")

✅ `숫자가 아닙니다` · `다음 줄로 갑니다` — 에러가 난 줄 아래(`바꿨습니다`)는 건너뛰고, `try` 밖의 줄은 실행됩니다

#### 🐍 문법 상자 · `requests.post` 의 `json=` 과 `timeout=`

```python
response = requests.post(url, json={"text": "알림"}, timeout=5)
print(response.status_code)          # 200 — 받았다는 뜻
```

| 쓰는 것 | 뜻 |
|---|---|
| `url` | 보낼 주소. 오늘은 `config["webhook_url"]` 에서 꺼낸다 |
| `json={"text": …}` | 딕셔너리를 JSON 으로 바꿔 요청 몸통에 담는다 (9/30) |
| `timeout=5` | 5초 안에 답이 없으면 기다리지 않고 에러를 낸다 |

⚠ `timeout` 이 없으면 서버가 답을 주지 않을 때 셀이 끝나지 않고 멈춰 있습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 알림을 보내는 notify 함수</font></h3></td></tr></table>

메시지를 알림 서버로 POST 하는 `notify` 함수를 완성하고, 테스트 알림을 한 번 보내시오. **알림 서버가 켜져 있어야** 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 노트북에 `200`, 알림 서버 터미널에 `[알림 받음] [테스트] 알림 연결 확인` |

**💡 힌트**

1. POST 는 9/30 에 쓴 `requests.post` 입니다.
2. `json=` 에는 `{"text": message}` 를 넣습니다. 서버가 `data["text"]` 로 꺼냅니다.
3. `ConnectionError` 가 나면 알림 서버 터미널이 켜져 있는지 봅니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
# ── 여기까지 ──


def notify(message, url):                                     # 메시지를 알림 서버로 보내는 함수
    # 1. response 라는 변수에 requests.post(url, json={"text": message}, timeout=5) 의 결과를 담으세요

    # 2. response.status_code 를 return 하세요



print(notify("[테스트] 알림 연결 확인", config["webhook_url"]))        # 켜진 서버로 보낸다 — 200 이 나와야 한다

#### 🐍 문법 상자 · `except Exception as e` — 에러를 잡아 이름 보기

```python
try:
    int("abc")
except Exception as e:
    print("잡음:", type(e).__name__)
# 잡음: ValueError
```

| 쓰는 것 | 잡는 것 |
|---|---|
| `except ValueError:` | 그 에러 하나만 (9/28) |
| `except Exception as e:` | 거의 모든 에러. 잡은 에러가 `e` 에 담긴다 |
| `type(e).__name__` | 에러의 이름 — `ValueError` · `ConnectionError` |

⚠ 에러를 잡고 아무것도 안 하면 에러가 났다는 사실이 드러나지 않습니다. `[알림 실패]` 처럼 반드시 출력하거나 기록합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 꺼진 서버로 보내 보기</font></h3></td></tr></table>

아무 서버도 없는 주소로 알림을 보내 보시오. 에러로 멈추지 않도록 `try` / `except` 로 감싸 에러 이름을 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림 실패] ConnectionError` · `보고서는 그대로 저장했습니다` |

**💡 힌트**

1. `try:` 아래에 보내는 줄을, `except Exception as e:` 아래에 출력하는 줄을 씁니다.
2. 에러 이름은 `type(e).__name__` 입니다.
3. `try` 없이 한 번 실행해 보면 빨간 에러로 멈추는 것을 볼 수 있습니다.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
url_off = "http://127.0.0.1:5999/alert"                      # 아무 서버도 없는 주소 — 알림 서버가 꺼진 상황
# ── 여기까지 ──

# 1. try 안에서 notify("[테스트] 꺼진 서버", url_off) 를 실행하세요

# 2. except Exception as e: 로 에러를 잡아 "[알림 실패]" 와 type(e).__name__ 을 출력하세요


print("보고서는 그대로 저장했습니다")                            # 에러가 나도 이 줄까지 옵니다

#### 🐍 문법 상자 · `try` 안의 `return`

```python
def safe_int(text):
    try:
        return int(text)
    except ValueError:
        return None

print(safe_int("5"), safe_int("abc"))
# 5 None
```

| 들어온 값 | 실행되는 곳 | 리턴하는 값 |
|---|---|---|
| `"5"` | `try` 안의 `return` | `5` |
| `"abc"` | 에러가 나서 `except` 안의 `return` | `None` |

⚠ 실패해도 함수가 값을 리턴하므로, 부른 쪽은 멈추지 않고 다음 줄로 갑니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 실패해도 값을 리턴하는 notify_safe</font></h3></td></tr></table>

`try` / `except` 를 함수 안으로 옮겨, 보내지면 `True`, 실패하면 `False` 를 리턴하는 `notify_safe` 를 완성하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `True` · `[알림 실패] ConnectionError` · `False` |

**💡 힌트**

1. `try` 안에서 보낸 뒤 `return True` 합니다.
2. `except` 안의 출력 줄은 미리 채워 두었습니다. 그 위에 `except` 줄을, 아래에 `return False` 를 씁니다.
3. 이제 부르는 쪽은 `try` 를 쓰지 않아도 됩니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
url_off = "http://127.0.0.1:5999/alert"                      # 꺼진 서버
# ── 여기까지 ──


def notify_safe(message, url):                                # 실패해도 멈추지 않는 알림 함수
    # 1. try 안에서 requests.post(url, json={"text": message}, timeout=5) 를 보내고 True 를 return 하세요

    # 2. except Exception as e: 를 쓰고, 아래 출력 줄 다음에 False 를 return 하세요

        # ── 미리 채운 줄 — except 안에 들어가도록 들여쓰기가 맞춰져 있습니다 ──
        print("[알림 실패]", type(e).__name__)                    # 에러 이름을 알린다 — 숨기지 않는다



print(notify_safe("[테스트] 켜진 서버", config["webhook_url"]))      # 켜진 서버 — True
print(notify_safe("[테스트] 꺼진 서버", url_off))                    # 꺼진 서버 — False

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · notifier.py 만들기</font></h3></td></tr></table>

오늘의 산출물 **`notifier.py`** 를 만드시오. 2교시의 `load_config` · `needs_approval` 은 옮겨 두었습니다. `notify` 의 빈칸만 채웁니다.

1. 아래 셀을 채워 실행하면 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing notifier.py`, 확인 셀은 `False` · `True` · `[알림 실패] ConnectionError` · `False` |

**💡 힌트**

1. 문제 3-4 의 `notify_safe` 와 같은 내용입니다. 이름만 `notify` 입니다.
2. `needs_approval` 은 모르는 위험도를 high 로 봅니다(`RANK.get(…, 3)`). 놓치는 것보다 묻는 게 낫습니다.
3. 확인 셀에서 오류가 나면 문제 셀을 고쳐 다시 실행하고, **확인 셀도 다시** 실행합니다.

In [ ]:
%%writefile notifier.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# 꼭 있어야 하는 키 네 개
REQUIRED = ["model", "approve_severity", "report_folder", "webhook_url"]
RANK = {"low": 1, "medium": 2, "high": 3}                     # 위험도 순위 — 클수록 위험하다


def load_config(path):                                        # 설정을 읽고 빠진 키가 없는지 검사하는 함수
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    for key in REQUIRED:                                      # 꼭 있어야 하는 키를 하나씩
        if key not in config:                                 # 설정에 그 키가 없으면
            print("[설정 오류] 빠진 키:", key)                       # 어떤 키가 빠졌는지 알린다
            return None                                       # None 을 리턴하고 여기서 끝낸다
    return config                                             # 다 있으면 설정을 리턴한다


def needs_approval(severity, config):                         # 사람에게 확인받을 위험도인지 판정하는 함수
    level = RANK.get(severity.lower(), 3)                     # 모르는 위험도는 high 로 본다 — 놓치는 것보다 묻는 게 낫다
    return level >= RANK[config["approve_severity"]]          # 기준 이상이면 True


def notify(message, url):                                     # 메시지를 알림 서버로 보내는 함수
    # 1. try 안에서 url 로 {"text": message} 를 POST 하고(timeout=5) True 를 return 하세요

    # 2. except Exception as e: 를 쓰고, 아래 출력 줄 다음에 False 를 return 하세요

        # ── 미리 채운 줄 — except 안에 들어가도록 들여쓰기가 맞춰져 있습니다 ──
        print("[알림 실패]", type(e).__name__)                    # 에러 이름을 알린다 — 숨기지 않는다



In [ ]:
# 확인 셀 — 그대로 실행합니다
import importlib                                              # 파일을 다시 읽는 도구
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(notifier)                                    # notifier.py 를 고쳤으면 다시 읽는다

config = notifier.load_config("config.json")                  # 설정을 검사하며 읽는다
# 기준이 high 라 False
print("medium 은 사람에게 묻나:", notifier.needs_approval("medium", config))
# 켜진 서버 — True
print("켜진 서버:", notifier.notify("[테스트] notifier.py 확인", config["webhook_url"]))
# 꺼진 서버 — False
print("꺼진 서버:", notifier.notify("[테스트] 꺼진 서버", "http://127.0.0.1:5999/alert"))

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

#### 🐍 문법 상자 · `"a"` 모드는 뒤에 이어 씁니다 (10/7 오전)

```python
with open("alert_failures.log", "a", encoding="utf-8") as f:
    f.write("[보고서] … 저장\n")       # 실행할 때마다 한 줄씩 늘어난다
```

| 모드 | 파일이 있으면 | 오늘 쓰는 곳 |
|---|---|---|
| `"w"` | 지우고 새로 쓴다 | `config.json` · 보고서 |
| `"a"` | 뒤에 이어 쓴다 | 실패 기록 — 쌓여야 한다 |

⚠ `"a"` 로 연 파일은 셀을 다시 실행할 때마다 줄이 늘어납니다. 줄 수가 2보다 많아도 맞습니다.

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-6 · 알림 실패를 파일에 쌓아 두기</font></h3></td></tr></table>

알림이 실패하면 그 메시지를 `alert_failures.log` 에 한 줄씩 **이어 쓰는** `notify_logged` 를 완성하시오. 서버가 다시 켜지면 이 파일을 보고 다시 보낼 수 있습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림 실패] ConnectionError` 두 번과 `실패 기록 2 줄` (다시 실행하면 줄 수가 늘어납니다) |

**💡 힌트**

1. 쌓아야 하므로 `"w"` 가 아니라 `"a"` 입니다.
2. 줄 끝에 `\n` 을 붙입니다.
3. 줄 수는 `len(f.readlines())` 로 셉니다.

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = notifier.load_config("config.json")
url_off = "http://127.0.0.1:5999/alert"                      # 꺼진 서버
# ── 여기까지 ──


def notify_logged(message, url):                              # 실패하면 파일에 남기는 알림 함수
    if notifier.notify(message, url):                         # 보내졌으면
        return True                                           # 기록할 것이 없다
    # 1. "alert_failures.log" 를 "a" 모드로 열어 f"{message}\n" 을 이어 쓰세요
    # 2. False 를 return 하세요


notify_logged("[보고서] daily_report_20261008.md 저장", url_off)   # 꺼진 서버로 — 실패 기록 1줄
notify_logged("[보고서] daily_report_20261009.md 저장", url_off)   # 꺼진 서버로 — 실패 기록 1줄 더

# 3. "alert_failures.log" 를 읽어 줄 수를 「실패 기록 2줄」 처럼 출력하세요


### 📋 3교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 알림을 보낸다 | `requests.post(url, json={"text": message}, timeout=5)` |
| 실패를 잡는다 | `try:` … `except Exception as e:` |
| 무엇이 실패했나 | `type(e).__name__` |
| 실패해도 이어 간다 | `return False` — 부른 쪽은 다음 줄로 간다 |

알림 하나가 실패해도 보고서는 남습니다. 실패는 숨기지 않고 출력합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · 하나로 잇기 — pipeline.py</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **오케스트레이션 (orchestration)** | 여러 단계를 정해진 순서대로 실행하도록 한 곳에서 관리하는 것 |
| **엔트리 포인트 (entry point)** | `python 파일.py` 로 실행하면 어디서부터 시작하나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 오케스트레이션 →
- 엔트리 포인트 →

### 왜 필요한가

1. 지금은 보고서 하나를 만들려면 노트북 셀을 **순서대로 여러 개** 눌러야 합니다. 하나 빠뜨리면 `NameError` 가 납니다.
2. 단계를 함수로 묶고, 함수 하나(`run_pipeline`)가 **순서대로 부르게** 합니다. 이것이 오케스트레이션입니다.
3. 마지막에 `pipeline.py` 파일로 만들면 노트북 없이 **터미널 명령 한 번**으로 돕니다. 10/2 에 배운 스케줄러로 매일 아침 돌릴 수도 있습니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 run_pipeline 이 단계를 차례로 부릅니다</mark>

```
run_pipeline("config.json", today)
  ├─ notifier.load_config        설정 읽기 · 검사          2교시
  ├─ run_report(events, today)   요약 → 정렬 → 총평 → 저장   어제 + 4-2   (LLM 두 번)
  ├─ needs_approval 로 세기       사람 확인이 필요한 건수     2교시
  └─ notifier.notify             알림 · 실패해도 계속         3교시
```

| 단계 | 넘겨받는 것 | 넘겨주는 것 |
|---|---|---|
| `load_config` | 파일 이름 | `config` 딕셔너리 |
| `run_report` | 경보 · 날짜 | `{"filename": …, "summaries": …}` |
| `notify` | 메시지 · 주소 | `True` / `False` |

앞 단계가 `return` 한 값이 다음 단계의 입력입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def count_high(levels):
    n = 0
    for level in levels:
        if level == "high":
            n = n + 1
    return n

def make_message(n):
    return f"high {n}건"

print(make_message(count_high(["high", "low", "high"])))
```

막히면 바로 위 `4.1 run_pipeline 이 단계를 차례로 부릅니다` 설명을 다시 봅니다.

In [ ]:
def count_high(levels):
    n = 0
    for level in levels:
        if level == "high":
            n = n + 1
    return n

def make_message(n):
    return f"high {n}건"

print(make_message(count_high(["high", "low", "high"])))

✅ `high 2건` — 안쪽 `count_high` 가 먼저 돌아 2 를 리턴하고, 그 값이 `make_message` 에 들어갑니다

#### 🐍 문법 상자 · `print` 는 화면에, `return` 은 다음 함수에

```python
def show():
    print("보고서 저장")

def give():
    return "보고서 저장"

x = show()
y = give()
print(x, y)
# 보고서 저장
# None 보고서 저장
```

| 함수 | 화면 | 리턴하는 값 |
|---|---|---|
| `show()` | `보고서 저장` | `None` — 아무것도 넘기지 않는다 |
| `give()` | 없음 | `"보고서 저장"` — 변수에 담아 다음 함수에 넘길 수 있다 |

⚠ 함수 사이를 잇는 것은 `return` 입니다. `print` 만 하면 다음 함수는 `None` 을 받습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 보고서 단계를 run_report 하나로 묶기</font></h3></td></tr></table>

어제 셀 여러 개로 하던 요약 → 정렬 → 총평 → 보고서 → 저장을 함수 `run_report` 하나로 묶으시오. **한 번 실행에 LLM 을 두 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `daily_report_20261008.md · 6 건` |

**💡 힌트**

1. 함수 이름 앞에 파일 이름(`event_summarizer.` · `report_generator.`)을 붙입니다.
2. 리턴할 것이 둘이라 딕셔너리 하나에 담아 `return` 합니다.
3. 빈 줄만 나오거나 총평이 「만들지 못했습니다」면 한도에 걸린 것입니다. 1분 뒤 다시 실행합니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1008.json", encoding="utf-8") as f:        # 준비 셀이 저장한 경보 6건
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
# ── 여기까지 ──


def run_report(events, today):                                # 요약 → 정렬 → 총평 → 저장을 묶은 함수
    # 1. summaries 에 event_summarizer.summarize_events(events) 의 결과를 담으세요 — LLM 한 번

    # 2. sorted_summaries 에 event_summarizer.sort_by_risk(summaries) 의 결과를 담으세요

    # ── 미리 채워 둔 두 줄입니다 — 10/7 오후의 총평 · 보고서 함수 ──
    overview = report_generator.make_overview(sorted_summaries)   # 총평 — LLM 한 번
    report = report_generator.build_report(sorted_summaries, overview, today)   # 틀 · 숫자 · 총평으로 보고서를 만든다
    # ── 여기까지 ──
    # 3. filename 에 report_generator.save_report(report, today) 의 결과를 담으세요

    # 4. {"filename": filename, "summaries": sorted_summaries} 를 return 하세요



result = run_report(events, "2026-10-08")                     # 보고서를 만든다 — LLM 두 번
print(result["filename"], "·", len(result["summaries"]), "건")  # 저장한 파일 이름과 요약 건수

#### 🐍 문법 상자 · 값 여러 개는 딕셔너리 하나로 리턴합니다

```python
def run():
    return {"filename": "daily_report_20261008.md", "count": 6}

result = run()
print(result["filename"], result["count"])
# daily_report_20261008.md 6
```

| 쓰는 것 | 뜻 |
|---|---|
| `return {"filename": …, "summaries": …}` | 결과 두 개를 이름표를 붙여 한 번에 리턴한다 |
| `result["filename"]` | 받은 쪽에서 이름으로 꺼낸다 |

⚠ `return` 은 한 번만 실행됩니다. 두 줄로 `return a` · `return b` 를 쓰면 두 번째 줄은 실행되지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · run_pipeline 으로 처음부터 알림까지</font></h3></td></tr></table>

설정 읽기 → 보고서 → 확인 건수 세기 → 알림을 차례로 부르는 `run_pipeline` 을 완성하시오. **한 번 실행에 LLM 을 두 번 부릅니다.** 알림 서버가 켜져 있어야 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[보고서] daily_report_20261008.md 저장 · 사람 확인 필요 1건` — 알림 서버 터미널에도 같은 줄이 찍힙니다(건수는 LLM 판단에 따라 다를 수 있습니다) |

**💡 힌트**

1. 설정이 `None` 이면 더 가지 않고 `return` 합니다.
2. 사람 확인 건수를 세는 세 줄은 미리 채워 두었습니다. 4-4 에서 직접 씁니다.
3. `llm_client.MODEL = config["model"]` 줄 덕분에 모델을 바꿀 때 `config.json` 만 고치면 됩니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다


def run_pipeline(config_path, today):                         # 설정 → 보고서 → 세기 → 알림을 차례로 부르는 함수
    # 1. config 에 notifier.load_config(config_path) 를 담고, None 이면 "[중단] config.json 을 고친 뒤 다시 실행하세요" 를 출력하고 return 하세요


    llm_client.MODEL = config["model"]                        # 모델 이름도 설정 파일에서 온다
    with open("events_1008.json", encoding="utf-8") as f:     # 오늘 처리할 경보 파일을 연다
        events = json.load(f)                                 # 경보 목록을 리스트로 읽는다
    result = run_report(events, today)                        # 문제 4-2 의 함수 — LLM 두 번

    approve = 0                                               # 사람 확인이 필요한 건수. 0 에서 시작
    # ── 미리 채워 둔 세 줄입니다 — 4-4 에서는 직접 씁니다 ──
    for s in result["summaries"]:                             # 정렬된 요약을 하나씩
        if notifier.needs_approval(s["risk_level"], config):  # 사람에게 확인받을 위험도면
            approve = approve + 1                             # 하나 센다
    # ── 여기까지 ──

    # 알림으로 보낼 한 줄
    message = f"[보고서] {result['filename']} 저장 · 사람 확인 필요 {approve}건"
    # 2. notifier.notify(message, config["webhook_url"]) 로 알림을 보내세요

    print(message)                                            # 보낸 알림을 화면에도 보여 준다 (미리 채운 줄)


run_pipeline("config.json", "2026-10-08")                     # 처음부터 끝까지 한 번 돌린다 — LLM 두 번

#### 🐍 문법 상자 · `if __name__ == "__main__":` — 직접 실행할 때만

```python
def run_pipeline(config_path, today):
    ...

if __name__ == "__main__":
    run_pipeline("config.json", str(date.today()))
```

| 어떻게 쓰나 | `__name__` 의 값 | 아래 줄 |
|---|---|---|
| 터미널에서 `python pipeline.py` | `"__main__"` | **실행된다** |
| 노트북에서 `import pipeline` | `"pipeline"` | 실행되지 않는다 — 함수만 불러온다 |
| `str(date.today())` | `"2026-10-08"` 꼴의 오늘 날짜 | |

⚠ 이 줄이 없으면 `import pipeline` 만 해도 LLM 을 두 번 부르고 알림까지 보냅니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · pipeline.py 만들기</font></h3></td></tr></table>

오늘의 산출물 **`pipeline.py`** 를 만드시오. 4-2 · 4-3 의 함수는 옮겨 두었습니다. **사람 확인 건수를 세는 세 줄**(4-3 에서 미리 채웠던 줄)과 맨 아래 **실행 시작 줄**을 씁니다.

1. 아래 셀을 채워 실행하면 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행합니다. `import` 만 하고 실행은 되지 않아야 맞습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing pipeline.py`, 확인 셀은 `불러오기만 했습니다 …` 와 `함수가 있나: True` |

**💡 힌트**

1. 세는 세 줄은 2교시 2-6 과 같은 모양입니다 — `for` · `if` · 1 더하기.
2. `if __name__ == "__main__":` 아래 줄은 들여 씁니다.
3. 날짜는 `str(date.today())` 로 오늘 날짜를 씁니다.
4. 확인 셀에서 `[보고서]` 줄이 나오면 `if __name__` 줄이 빠진 것입니다.

In [ ]:
%%writefile pipeline.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from datetime import date                                     # 오늘 날짜를 얻는 도구

import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다


def run_report(events, today):                                # 요약 → 정렬 → 총평 → 저장을 묶은 함수
    summaries = event_summarizer.summarize_events(events)     # 묶음으로 나눠 요약한다 — LLM
    # 위험도순으로 정렬한다
    sorted_summaries = event_summarizer.sort_by_risk(summaries)
    # 총평을 받는다 — LLM 한 번
    overview = report_generator.make_overview(sorted_summaries)
    # 틀 · 숫자 · 총평으로 보고서를 만든다
    report = report_generator.build_report(sorted_summaries, overview, today)
    filename = report_generator.save_report(report, today)    # 날짜가 든 파일로 저장한다
    # 결과 둘을 딕셔너리 하나로 리턴한다
    return {"filename": filename, "summaries": sorted_summaries}


def run_pipeline(config_path, today):                         # 설정 → 보고서 → 세기 → 알림을 차례로 부르는 함수
    config = notifier.load_config(config_path)                # 설정을 검사하며 읽는다
    if config is None:                                        # 설정에 빠진 키가 있으면
        print("[중단] config.json 을 고친 뒤 다시 실행하세요")             # 왜 멈췄는지 알린다
        return                                                # 더 가지 않고 함수를 끝낸다

    llm_client.MODEL = config["model"]                        # 모델 이름도 설정 파일에서 온다
    with open("events_1008.json", encoding="utf-8") as f:     # 오늘 처리할 경보 파일을 연다
        events = json.load(f)                                 # 경보 목록을 리스트로 읽는다
    result = run_report(events, today)                        # 보고서를 만든다 — LLM 두 번

    approve = 0                                               # 사람 확인이 필요한 건수. 0 에서 시작
    # 1. result["summaries"] 를 돌면서 notifier.needs_approval(s["risk_level"], config) 가 True 면 approve 에 1 을 더하세요



    # 알림으로 보낼 한 줄
    message = f"[보고서] {result['filename']} 저장 · 사람 확인 필요 {approve}건"
    notifier.notify(message, config["webhook_url"])           # 알림 — 실패해도 멈추지 않는다
    print(message)                                            # 보낸 알림을 화면에도 보여 준다


# 2. 이 파일을 python pipeline.py 로 직접 실행할 때만 아래 줄이 돌도록 if __name__ == "__main__": 을 쓰세요

    # 3. run_pipeline("config.json", str(date.today())) 를 실행하세요


In [ ]:
# 확인 셀 — 그대로 실행합니다
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

print("불러오기만 했습니다. 위에 [보고서] 줄이 없으면 맞습니다")                     # [보고서] 줄이 없어야 맞다
print("함수가 있나:", "run_pipeline" in dir(pipeline))        # dir() 은 그 파일 안의 이름 목록

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 터미널에서 명령 한 번으로 돌리기</font></h3></td></tr></table>

노트북을 쓰지 않고 터미널에서 `pipeline.py` 를 실행하시오. **LLM 을 두 번 부릅니다.**

1. 알림 서버 터미널은 그대로 둡니다.
2. **`agent_core` 폴더를 오른쪽 클릭 › Open in Integrated Terminal** 로 터미널을 하나 더 엽니다.
3. 새 터미널에 명령을 입력합니다. 코드 셀은 없습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 새 터미널에 `[보고서] daily_report_오늘날짜.md 저장 · 사람 확인 필요 1건`, 알림 서버 터미널에 `[알림 받음] …` |

**💡 힌트**

1. 실행할 파일 이름을 `python` 뒤에 씁니다.
2. `No such file` 이 나오면 터미널이 `agent_core` 가 아닌 곳에서 열린 것입니다.
3. 왼쪽 목록에 오늘 날짜의 보고서 파일이 새로 생겼는지 봅니다.

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

#### 🐍 문법 상자 · 폴더 만들기와 경로 잇기

```python
os.makedirs("reports", exist_ok=True)                        # 이미 있어도 에러가 나지 않는다
print(os.path.join("reports", "daily_report_20261008.md"))
# Windows: reports\daily_report_20261008.md   ·   Mac: reports/daily_report_20261008.md
```

| 쓰는 것 | 하는 일 |
|---|---|
| `os.makedirs(폴더, exist_ok=True)` | 폴더를 만든다. 있으면 그냥 지나간다 |
| `os.path.join(폴더, 파일)` | 그 컴퓨터에 맞는 구분 기호로 경로를 잇는다 |
| `shutil.copy(원본, 새 경로)` | 파일을 복사한다 |

⚠ `"reports/" + filename` 처럼 직접 이으면 Windows 와 Mac 에서 모양이 달라집니다. `os.path.join` 을 씁니다.

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-6 · 보고서를 설정의 폴더에 모으기</font></h3></td></tr></table>

`config.json` 의 `report_folder` 폴더를 만들고, 보고서를 그 안으로 복사하시오. 폴더 이름도 설정에서 옵니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `reports/daily_report_20261008.md` (Windows 는 `reports\daily_report_20261008.md`) 와 `['daily_report_20261008.md']` |

**💡 힌트**

1. 폴더 이름을 코드에 쓰지 않고 `config["report_folder"]` 로 꺼냅니다.
2. `exist_ok=True` 가 있으면 다시 실행해도 에러가 나지 않습니다.
3. `pipeline.py` 의 `run_report` 에 이 줄들을 넣으면 매일 보고서가 한 폴더에 쌓입니다.

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import shutil                                                 # 파일을 복사 · 이동하는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
filename = "daily_report_20261008.md"                         # 문제 4-2 가 만든 보고서
# ── 여기까지 ──

# 1. folder 에 config["report_folder"] 를 담고, os.makedirs(folder, exist_ok=True) 로 폴더를 만드세요
# 2. new_path 에 os.path.join(folder, filename) 을 담으세요
# 3. shutil.copy(filename, new_path) 로 보고서를 복사하세요
# 4. new_path 와 os.listdir(folder) 를 출력하세요


### 📋 4교시 한눈에

| 단계 | 함수 | 파일 |
|---|---|---|
| 설정 읽기 · 검사 | `load_config` | `notifier.py` |
| 요약 → 정렬 → 총평 → 저장 | `run_report` | `pipeline.py` |
| 확인 건수 세기 | `needs_approval` | `notifier.py` |
| 알림 | `notify` | `notifier.py` |
| 전체 실행 순서 | `run_pipeline` | `pipeline.py` |

오전 산출물은 `agent_core/config.json` · `notifier.py` · `pipeline.py` 입니다. 오후에는 이 코드를 **리뷰하고, 테스트하고, 에러를 고칩니다.**

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 2-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}   # 모델 이름 · 사람에게 물을 위험도
# ── 여기까지 ──

# 1. config 의 "report_folder" 키에 "reports" 를 담으세요
config["report_folder"] = "reports"                           # 보고서를 모을 폴더 이름

# 2. config 의 "webhook_url" 키에 "http://127.0.0.1:5001/alert" 를 담으세요
config["webhook_url"] = "http://127.0.0.1:5001/alert"         # 알림을 보낼 주소

# 3. config 를 "config.json" 파일에 저장하세요 (json.dump, ensure_ascii=False, indent=2)
with open("config.json", "w", encoding="utf-8") as f:         # 설정 파일을 쓰기로 연다
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다

print("config.json 을 저장했습니다 · 키", len(config), "개")           # 키가 몇 개인지 본다

### 정답 2-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# 1. "config.json" 을 읽기로 열어 config 에 담으세요
with open("config.json", encoding="utf-8") as f:              # 설정 파일을 읽기로 연다
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다

# 2. config 에서 모델 이름과 알림 주소를 꺼내 출력하세요
print("모델:", config["model"])                                 # 키로 값을 꺼낸다
print("알림 주소:", config["webhook_url"])                        # 키로 값을 꺼낸다

### 정답 2-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
REQUIRED = ["model", "approve_severity", "report_folder", "webhook_url"]   # 꼭 있어야 하는 키 네 개
broken = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}   # 키 두 개가 빠진 설정
with open("config_broken.json", "w", encoding="utf-8") as f:  # 연습용 — 키가 빠진 설정 파일
    json.dump(broken, f)                                      # 빠진 설정을 저장한다
# ── 여기까지 ──


def load_config(path):                                        # 설정을 읽고 빠진 키가 없는지 검사하는 함수
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    # 1. REQUIRED 의 키를 하나씩 꺼내, config 에 없으면 "[설정 오류] 빠진 키:" 와 키 이름을 출력하고 None 을 return 하세요
    for key in REQUIRED:                                      # 꼭 있어야 하는 키를 하나씩
        if key not in config:                                 # 설정에 그 키가 없으면
            print("[설정 오류] 빠진 키:", key)                       # 어떤 키가 빠졌는지 알린다
            return None                                       # None 을 리턴하고 여기서 끝낸다
    # 2. 다 있으면 config 를 return 하세요
    return config                                             # 다 있으면 설정을 리턴한다


print(load_config("config.json"))                             # 키가 다 있는 설정 — 딕셔너리가 나온다
print(load_config("config_broken.json"))                      # 키가 빠진 설정 — None 이 나온다

### 정답 2-5

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
RANK = {"low": 1, "medium": 2, "high": 3}                    # 위험도 순위 — 클수록 위험하다
# ── 여기까지 ──


def needs_approval(severity, config):                         # 사람에게 확인받을 위험도인지 판정하는 함수
    # 1. level 이라는 변수에 RANK 에서 severity.lower() 의 순위를 꺼내 담으세요
    level = RANK[severity.lower()]                            # 이 경보의 순위
    # 2. limit 이라는 변수에 RANK 에서 config["approve_severity"] 의 순위를 꺼내 담으세요
    limit = RANK[config["approve_severity"]]                  # 설정이 정한 기준 순위
    # 3. level 이 limit 보다 크거나 같은지(>=)를 return 하세요
    return level >= limit                                     # 기준 이상이면 True


for severity in ["low", "medium", "High"]:                    # 위험도 셋으로 시험한다 — 대문자가 섞여 있다
    print(severity, needs_approval(severity, config))         # 위험도와 판정 결과

### 정답 ⭐2-6

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
RANK = {"low": 1, "medium": 2, "high": 3}                     # 위험도 순위 — 클수록 위험하다


def needs_approval(severity, config):                         # 문제 2-5 의 함수
    # 순위가 기준 이상이면 True
    return RANK[severity.lower()] >= RANK[config["approve_severity"]]


severities = ["low", "medium", "high", "medium", "low", "high"]   # 오늘 판정한 위험도라고 가정합니다
# ── 여기까지 ──

# 1. config.json 을 읽어 config 에 담고, approve_severity 를 "medium" 으로 바꾼 것을 "config_medium.json" 으로 저장하세요
# 2. 두 파일을 하나씩 읽어, needs_approval 이 True 인 개수를 세세요
# 3. 「config.json : 2건」 · 「config_medium.json : 4건」 처럼 출력하세요
with open("config.json", encoding="utf-8") as f:              # 설정 파일을 읽기로 연다
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
config["approve_severity"] = "medium"                         # 기준을 medium 으로 낮춘다
with open("config_medium.json", "w", encoding="utf-8") as f:  # 새 설정 파일로 저장한다 — config.json 은 그대로
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다

for path in ["config.json", "config_medium.json"]:            # 두 설정 파일을 차례로
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for severity in severities:                               # 위험도를 하나씩
        if needs_approval(severity, config):                  # 이 설정에서 확인이 필요하면
            count = count + 1                                 # 하나 센다
    print(path, ":", count, "건")                              # 파일마다 건수

### 정답 3-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
# ── 여기까지 ──


def notify(message, url):                                     # 메시지를 알림 서버로 보내는 함수
    # 1. response 라는 변수에 requests.post(url, json={"text": message}, timeout=5) 의 결과를 담으세요
    # 메시지를 JSON 으로 담아 POST — 5초만 기다린다
    response = requests.post(url, json={"text": message}, timeout=5)
    # 2. response.status_code 를 return 하세요
    return response.status_code                               # 200 이면 받았다는 뜻


print(notify("[테스트] 알림 연결 확인", config["webhook_url"]))        # 켜진 서버로 보낸다 — 200 이 나와야 한다

### 정답 3-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
url_off = "http://127.0.0.1:5999/alert"                      # 아무 서버도 없는 주소 — 알림 서버가 꺼진 상황
# ── 여기까지 ──

# 1. try 안에서 notify("[테스트] 꺼진 서버", url_off) 를 실행하세요
try:                                                          # 아래 줄을 해 본다
    notify("[테스트] 꺼진 서버", url_off)                            # 꺼진 서버로 보낸다 — 여기서 에러가 난다
# 2. except Exception as e: 로 에러를 잡아 "[알림 실패]" 와 type(e).__name__ 을 출력하세요
except Exception as e:                                        # 에러가 나면 e 에 담아 여기로 온다
    print("[알림 실패]", type(e).__name__)                        # 에러 이름을 알린다 — 숨기지 않는다

print("보고서는 그대로 저장했습니다")                            # 에러가 나도 이 줄까지 옵니다

### 정답 3-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
url_off = "http://127.0.0.1:5999/alert"                      # 꺼진 서버
# ── 여기까지 ──


def notify_safe(message, url):                                # 실패해도 멈추지 않는 알림 함수
    # 1. try 안에서 requests.post(url, json={"text": message}, timeout=5) 를 보내고 True 를 return 하세요
    try:                                                      # 아래 줄을 해 본다
        requests.post(url, json={"text": message}, timeout=5)  # 메시지를 POST 한다 — 5초만 기다린다
        return True                                           # 보냈다
    # 2. except Exception as e: 를 쓰고, 아래 출력 줄 다음에 False 를 return 하세요
    except Exception as e:                                    # 에러가 나면 e 에 담아 여기로 온다
        print("[알림 실패]", type(e).__name__)                    # 에러 이름을 알린다 — 숨기지 않는다
        return False                                          # 못 보냈다 — 부른 쪽은 계속 간다


print(notify_safe("[테스트] 켜진 서버", config["webhook_url"]))      # 켜진 서버 — True
print(notify_safe("[테스트] 꺼진 서버", url_off))                    # 꺼진 서버 — False

### 정답 3-5

In [ ]:
%%writefile notifier.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# 꼭 있어야 하는 키 네 개
REQUIRED = ["model", "approve_severity", "report_folder", "webhook_url"]
RANK = {"low": 1, "medium": 2, "high": 3}                     # 위험도 순위 — 클수록 위험하다


def load_config(path):                                        # 설정을 읽고 빠진 키가 없는지 검사하는 함수
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    for key in REQUIRED:                                      # 꼭 있어야 하는 키를 하나씩
        if key not in config:                                 # 설정에 그 키가 없으면
            print("[설정 오류] 빠진 키:", key)                       # 어떤 키가 빠졌는지 알린다
            return None                                       # None 을 리턴하고 여기서 끝낸다
    return config                                             # 다 있으면 설정을 리턴한다


def needs_approval(severity, config):                         # 사람에게 확인받을 위험도인지 판정하는 함수
    level = RANK.get(severity.lower(), 3)                     # 모르는 위험도는 high 로 본다 — 놓치는 것보다 묻는 게 낫다
    return level >= RANK[config["approve_severity"]]          # 기준 이상이면 True


def notify(message, url):                                     # 메시지를 알림 서버로 보내는 함수
    # 1. try 안에서 url 로 {"text": message} 를 POST 하고(timeout=5) True 를 return 하세요
    try:                                                      # 아래 줄을 해 본다
        requests.post(url, json={"text": message}, timeout=5)  # 메시지를 POST 한다 — 5초만 기다린다
        return True                                           # 보냈다
    # 2. except Exception as e: 를 쓰고, 아래 출력 줄 다음에 False 를 return 하세요
    except Exception as e:                                    # 에러가 나면 e 에 담아 여기로 온다
        print("[알림 실패]", type(e).__name__)                    # 에러 이름을 알린다 — 숨기지 않는다
        return False                                          # 못 보냈다 — 부른 쪽은 계속 간다

### 정답 ⭐3-6

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = notifier.load_config("config.json")
url_off = "http://127.0.0.1:5999/alert"                      # 꺼진 서버
# ── 여기까지 ──


def notify_logged(message, url):                              # 실패하면 파일에 남기는 알림 함수
    if notifier.notify(message, url):                         # 보내졌으면
        return True                                           # 기록할 것이 없다
    # 1. "alert_failures.log" 를 "a" 모드로 열어 f"{message}\n" 을 이어 쓰세요
    # 실패 기록 파일을 이어 쓰기로 연다
    with open("alert_failures.log", "a", encoding="utf-8") as f:
        f.write(f"{message}\n")                               # 못 보낸 메시지를 한 줄 남긴다
    # 2. False 를 return 하세요
    return False                                              # 못 보냈다 — 부른 쪽은 계속 간다


notify_logged("[보고서] daily_report_20261008.md 저장", url_off)   # 꺼진 서버로 — 실패 기록 1줄
notify_logged("[보고서] daily_report_20261009.md 저장", url_off)   # 꺼진 서버로 — 실패 기록 1줄 더

# 3. "alert_failures.log" 를 읽어 줄 수를 「실패 기록 2줄」 처럼 출력하세요
with open("alert_failures.log", encoding="utf-8") as f:       # 실패 기록을 읽기로 연다
    print("실패 기록", len(f.readlines()), "줄")                   # 줄 수 = 실패한 알림 수

### 정답 4-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1008.json", encoding="utf-8") as f:        # 준비 셀이 저장한 경보 6건
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
# ── 여기까지 ──


def run_report(events, today):                                # 요약 → 정렬 → 총평 → 저장을 묶은 함수
    # 1. summaries 에 event_summarizer.summarize_events(events) 의 결과를 담으세요 — LLM 한 번
    summaries = event_summarizer.summarize_events(events)     # 묶음으로 나눠 요약한다 — LLM
    # 2. sorted_summaries 에 event_summarizer.sort_by_risk(summaries) 의 결과를 담으세요
    # 위험도순으로 정렬한다
    sorted_summaries = event_summarizer.sort_by_risk(summaries)
    # ── 미리 채워 둔 두 줄입니다 — 10/7 오후의 총평 · 보고서 함수 ──
    overview = report_generator.make_overview(sorted_summaries)   # 총평 — LLM 한 번
    report = report_generator.build_report(sorted_summaries, overview, today)   # 틀 · 숫자 · 총평으로 보고서를 만든다
    # ── 여기까지 ──
    # 3. filename 에 report_generator.save_report(report, today) 의 결과를 담으세요
    filename = report_generator.save_report(report, today)    # 날짜가 든 파일로 저장한다
    # 4. {"filename": filename, "summaries": sorted_summaries} 를 return 하세요
    # 결과 둘을 딕셔너리 하나로 리턴한다
    return {"filename": filename, "summaries": sorted_summaries}


result = run_report(events, "2026-10-08")                     # 보고서를 만든다 — LLM 두 번
print(result["filename"], "·", len(result["summaries"]), "건")  # 저장한 파일 이름과 요약 건수

### 정답 4-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다
importlib.reload(event_summarizer)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다
importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다


def run_pipeline(config_path, today):                         # 설정 → 보고서 → 세기 → 알림을 차례로 부르는 함수
    # 1. config 에 notifier.load_config(config_path) 를 담고, None 이면 "[중단] config.json 을 고친 뒤 다시 실행하세요" 를 출력하고 return 하세요
    config = notifier.load_config(config_path)                # 설정을 검사하며 읽는다
    if config is None:                                        # 설정에 빠진 키가 있으면
        print("[중단] config.json 을 고친 뒤 다시 실행하세요")             # 왜 멈췄는지 알린다
        return                                                # 더 가지 않고 함수를 끝낸다

    llm_client.MODEL = config["model"]                        # 모델 이름도 설정 파일에서 온다
    with open("events_1008.json", encoding="utf-8") as f:     # 오늘 처리할 경보 파일을 연다
        events = json.load(f)                                 # 경보 목록을 리스트로 읽는다
    result = run_report(events, today)                        # 문제 4-2 의 함수 — LLM 두 번

    approve = 0                                               # 사람 확인이 필요한 건수. 0 에서 시작
    # ── 미리 채워 둔 세 줄입니다 — 4-4 에서는 직접 씁니다 ──
    for s in result["summaries"]:                             # 정렬된 요약을 하나씩
        if notifier.needs_approval(s["risk_level"], config):  # 사람에게 확인받을 위험도면
            approve = approve + 1                             # 하나 센다
    # ── 여기까지 ──

    # 알림으로 보낼 한 줄
    message = f"[보고서] {result['filename']} 저장 · 사람 확인 필요 {approve}건"
    # 2. notifier.notify(message, config["webhook_url"]) 로 알림을 보내세요
    notifier.notify(message, config["webhook_url"])           # 알림 — 실패해도 멈추지 않는다
    print(message)                                            # 보낸 알림을 화면에도 보여 준다 (미리 채운 줄)


run_pipeline("config.json", "2026-10-08")                     # 처음부터 끝까지 한 번 돌린다 — LLM 두 번

### 정답 4-4

In [ ]:
%%writefile pipeline.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from datetime import date                                     # 오늘 날짜를 얻는 도구

import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다


def run_report(events, today):                                # 요약 → 정렬 → 총평 → 저장을 묶은 함수
    summaries = event_summarizer.summarize_events(events)     # 묶음으로 나눠 요약한다 — LLM
    # 위험도순으로 정렬한다
    sorted_summaries = event_summarizer.sort_by_risk(summaries)
    # 총평을 받는다 — LLM 한 번
    overview = report_generator.make_overview(sorted_summaries)
    # 틀 · 숫자 · 총평으로 보고서를 만든다
    report = report_generator.build_report(sorted_summaries, overview, today)
    filename = report_generator.save_report(report, today)    # 날짜가 든 파일로 저장한다
    # 결과 둘을 딕셔너리 하나로 리턴한다
    return {"filename": filename, "summaries": sorted_summaries}


def run_pipeline(config_path, today):                         # 설정 → 보고서 → 세기 → 알림을 차례로 부르는 함수
    config = notifier.load_config(config_path)                # 설정을 검사하며 읽는다
    if config is None:                                        # 설정에 빠진 키가 있으면
        print("[중단] config.json 을 고친 뒤 다시 실행하세요")             # 왜 멈췄는지 알린다
        return                                                # 더 가지 않고 함수를 끝낸다

    llm_client.MODEL = config["model"]                        # 모델 이름도 설정 파일에서 온다
    with open("events_1008.json", encoding="utf-8") as f:     # 오늘 처리할 경보 파일을 연다
        events = json.load(f)                                 # 경보 목록을 리스트로 읽는다
    result = run_report(events, today)                        # 보고서를 만든다 — LLM 두 번

    approve = 0                                               # 사람 확인이 필요한 건수. 0 에서 시작
    # 1. result["summaries"] 를 돌면서 notifier.needs_approval(s["risk_level"], config) 가 True 면 approve 에 1 을 더하세요
    for s in result["summaries"]:                             # 정렬된 요약을 하나씩
        if notifier.needs_approval(s["risk_level"], config):  # 사람에게 확인받을 위험도면
            approve = approve + 1                             # 하나 센다

    # 알림으로 보낼 한 줄
    message = f"[보고서] {result['filename']} 저장 · 사람 확인 필요 {approve}건"
    notifier.notify(message, config["webhook_url"])           # 알림 — 실패해도 멈추지 않는다
    print(message)                                            # 보낸 알림을 화면에도 보여 준다


# 2. 이 파일을 python pipeline.py 로 직접 실행할 때만 아래 줄이 돌도록 if __name__ == "__main__": 을 쓰세요
if __name__ == "__main__":                                    # python pipeline.py 로 직접 실행할 때만
    # 3. run_pipeline("config.json", str(date.today())) 를 실행하세요
    run_pipeline("config.json", str(date.today()))            # 오늘 날짜로 처음부터 끝까지 돌린다

### 정답 4-5

💻 **터미널에 입력합니다** — 알림 서버가 켜진 터미널이 아니라 **새로 연 터미널**입니다.

```bash
python pipeline.py
```

새 터미널에 `[보고서] daily_report_오늘날짜.md 저장 · 사람 확인 필요 1건` 이 보이고, 알림 서버 터미널에 `[알림 받음] [보고서] …` 가 보이면 됩니다.

### 정답 ⭐4-6

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import shutil                                                 # 파일을 복사 · 이동하는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:            # 2교시에 만든 설정 파일
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
filename = "daily_report_20261008.md"                         # 문제 4-2 가 만든 보고서
# ── 여기까지 ──

# 1. folder 에 config["report_folder"] 를 담고, os.makedirs(folder, exist_ok=True) 로 폴더를 만드세요
# 2. new_path 에 os.path.join(folder, filename) 을 담으세요
# 3. shutil.copy(filename, new_path) 로 보고서를 복사하세요
# 4. new_path 와 os.listdir(folder) 를 출력하세요
folder = config["report_folder"]                              # 폴더 이름도 설정에서 꺼낸다
os.makedirs(folder, exist_ok=True)                            # 폴더를 만든다. 있으면 그냥 지나간다
new_path = os.path.join(folder, filename)                     # 폴더와 파일 이름을 잇는다
shutil.copy(filename, new_path)                               # 보고서를 폴더 안으로 복사한다
print(new_path)                                               # 복사한 경로
print(os.listdir(folder))                                     # 폴더 안의 파일 목록